# §1 Individual (unfiltered) (v12 top-50, V4 filter)

Per-combo metrics and per-combo equity/drawdown curves on the
20% OOS test partition with no ML#2 filter. Sizing: `fixed_dollars_500` (risk $500 per trade, flat).

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

REPO = Path.cwd().resolve()
while not (REPO / 'src').exists() and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from scripts.evaluation._top_perf_common import (
    STARTING_EQUITY, POLICIES,
    apply_sizing, metrics_from_pnl, monte_carlo,
    load_setup,
    plot_indiv_equity, plot_indiv_dd,
    plot_combined_equity, plot_combined_dd,
    plot_ml2_indiv_equity, plot_ml2_indiv_dd,
    plot_ml2_combined_equity, plot_ml2_combined_dd,
    plot_mc_sims, plot_mc_pnl, plot_mc_sharpe, plot_mc_dd,
)

_ctx = load_setup(cost_per_contract_rt=0.0, top_strategies_path=REPO / 'evaluation' / 'top_strategies_v12_top50.json', version='v4')
bars            = _ctx['bars']
YEARS_SPAN      = _ctx['years_span']
strategies      = _ctx['strategies']
results_raw     = _ctx['results_raw']
combined_raw    = _ctx['combined_raw']
combos_ml2      = _ctx['combos_ml2']
s4_pnl_by_combo = _ctx['s4_pnl_by_combo']
ml2_portfolio   = _ctx['ml2_portfolio']


Top-K source: top_strategies_v12_top50.json


Test partition: 514,563 bars  2024-10-22 05:08:00 -> 2026-04-08 20:20:00
Years span: 1.461  (used to annualize Sharpe)
Loaded 50 strategies.
Loaded results_raw from cache (50 combos).
Combined unfiltered trades: 24,690
Loaded combos_ml2 from cache (50 combos).
ML2 portfolio trade counts: {'fixed_dollars_500': 681}


In [2]:
rows = []
for r in results_raw:
    if r['trades'].empty:
        for policy in POLICIES:
            rows.append({'combo_id': r['combo_id'], 'policy': policy,
                         **metrics_from_pnl(np.array([]), YEARS_SPAN, policy=policy)})
        continue
    t = r['trades'].sort_values('date', kind='mergesort')
    pnl_base = t['actual_pnl'].to_numpy(dtype=float)
    risk_base = t['dollar_risk'].to_numpy(dtype=float)
    r_mult = np.where(risk_base > 0, pnl_base / risk_base, 0.0)
    for policy in POLICIES:
        pnl = apply_sizing(pnl_base, risk_base, policy)
        rows.append({'combo_id': r['combo_id'], 'policy': policy,
                     **metrics_from_pnl(pnl, YEARS_SPAN, policy=policy, r=r_mult)})
perf1 = pd.DataFrame(rows)
perf1

,combo_id,policy,n_trades,trades_per_year,win_rate,total_pnl_dollars,total_return_pct,sharpe_ratio,max_drawdown_pct,max_drawdown_dollars
0,v11_2646,fixed_dollars_500,294,201.2,0.1667,-13728.96,-27.46,-1.2725,37.27,19084.91
1,v11_391,fixed_dollars_500,178,121.8,0.2809,1941.24,3.88,0.1850,19.32,10781.50
2,v11_28651,fixed_dollars_500,1155,790.5,0.3108,51687.11,103.37,1.4590,20.88,17189.95
3,v11_17782,fixed_dollars_500,401,274.5,0.3167,13377.70,26.76,0.7877,16.65,11583.26
4,v11_263,fixed_dollars_500,482,329.9,0.3444,-15576.70,-31.15,-0.9964,41.49,22657.72
5,v11_18020,fixed_dollars_500,434,297.1,0.2535,18997.97,38.00,1.1364,13.31,8208.27
6,v11_12101,fixed_dollars_500,323,221.1,0.2012,-3937.86,-7.88,-0.2685,24.68,14745.82
7,v11_27291,fixed_dollars_500,451,308.7,0.4390,3538.39,7.08,0.2624,20.07,13140.56
8,v11_3547,fixed_dollars_500,327,223.8,0.1743,32414.94,64.83,1.7434,10.12,6653.49
9,v11_25420,fixed_dollars_500,492,336.8,0.2053,4973.12,9.95,0.2886,19.39,12409.79


In [3]:
plot_indiv_equity(results_raw, 'fixed_dollars_500')

In [4]:
plot_indiv_dd(results_raw, 'fixed_dollars_500')